# Task 3: Impact Modeling

## Objective
This notebook develops statistical and machine learning models to:
1. Quantify the impact of socio-economic and digital factors on financial inclusion.
2. Predict financial inclusion outcomes.
3. Identify the most influential drivers for policy and strategic decision-making in Ethiopia.

## Key Questions
- Which variables most strongly affect financial inclusion?
- How accurately can we predict inclusion levels?
- What policy levers have the highest marginal impact?

## Imports Libraries

In [20]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from statsmodels.formula.api import ols
from statsmodels.tsa.statespace.sarimax import SARIMAX

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression

import warnings
warnings.filterwarnings("ignore")

plt.style.use("default")


## Load Processed Dataset

In [21]:
DATA_PATH = "../data/processed/ethiopia_fi_unified_data_enriched.csv"

df = pd.read_csv(DATA_PATH)

df.head()

,record_id,record_type,category,pillar,indicator,indicator_code,indicator_direction,value_numeric,value_text,value_type,...,impact_estimate,lag_months,evidence_basis,comparable_country,collected_by,collection_date,original_text,notes,_empty_col_34,_empty_col_35
0,REC_0001,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,22.0,NaN,percentage,...,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Baseline year,NaN,NaN,NaN
1,REC_0002,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,35.0,NaN,percentage,...,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN,NaN,NaN
2,REC_0003,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,46.0,NaN,percentage,...,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,NaN,NaN,NaN,NaN
3,REC_0004,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,56.0,NaN,percentage,...,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN,NaN,NaN
4,REC_0005,observation,NaN,ACCESS,Account Ownership Rate,ACC_OWNERSHIP,higher_better,36.0,NaN,percentage,...,NaN,NaN,NaN,Example_Trainee,2025-01-20,NaN,Gender disaggregated,NaN,NaN,NaN


## Data Preparation for Impact Modeling

### Assumptions

**Dataset contains:**

 - date (monthly or quarterly)

 - event_flag (0 = no event, 1 = event period)

 - event_name

 - Financial inclusion indicators (accounts, mobile money, agents, usage)

## Convert Date & Sort

In [24]:
df.columns = df.columns.str.strip()

# Find the column that contains "date" (case-insensitive)
date_col = [col for col in df.columns if col.lower() == "date"]
if not date_col:
    raise KeyError("No column named 'date' found in the DataFrame.")
date_col = date_col[0]

# Convert to datetime
df["date"] = pd.to_datetime(df[date_col])

# Sort by date and reset index
df = df.sort_values("date").reset_index(drop=True)

# Show info
df.info()

KeyError: "No column named 'date' found in the DataFrame."

## Train-Test Split & Scaling

In [12]:
# Only split if at least one target exists
if y_reg is not None or y_clf is not None:
    # Split for regression if available
    if y_reg is not None:
        X_train, X_test, y_train_reg, y_test_reg = train_test_split(
            X, y_reg, test_size=0.2, random_state=42
        )
    # Otherwise, split for classification
    elif y_clf is not None:
        X_train, X_test, y_train_clf, y_test_clf = train_test_split(
            X, y_clf, test_size=0.2, random_state=42
        )

    # Scale features
    scaler = StandardScaler()
    X_train_scaled = scaler.fit_transform(X_train)
    X_test_scaled = scaler.transform(X_test)
else:
    print("No valid target columns found. Cannot split or scale.")

No valid target columns found. Cannot split or scale.


## Baseline Model – Linear Regression

In [13]:
if 'X_train_scaled' in globals() and y_train_reg is not None:
    lin_reg = LinearRegression()
    lin_reg.fit(X_train_scaled, y_train_reg)

    y_pred_reg = lin_reg.predict(X_test_scaled)

    rmse = np.sqrt(mean_squared_error(y_test_reg, y_pred_reg))
    r2 = r2_score(y_test_reg, y_pred_reg)

    print("RMSE:", rmse)
    print("R^2:", r2)
else:
    print("Regression target not available. Skipping model training.")

Regression target not available. Skipping model training.


## Coefficient Analysis

In [15]:
if 'lin_reg' in globals() and hasattr(lin_reg, "coef_"):
    coefficients = pd.DataFrame({
        "Feature": X.columns,
        "Coefficient": lin_reg.coef_
    }).sort_values(by="Coefficient", ascending=False)

    print(coefficients.head(10))
else:
    print("Linear regression model not trained. Cannot show coefficients.")

Linear regression model not trained. Cannot show coefficients.


In [17]:
if 'coefficients' in globals() and not coefficients.empty:
    plt.figure(figsize=(8, 6))
    sns.barplot(
        data=coefficients.head(10),
        x="Coefficient",
        y="Feature"
    )
    plt.title("Top Positive Drivers of Financial Inclusion")
    plt.show()
else:
    print("Coefficients not available. Cannot plot feature importance.")

Coefficients not available. Cannot plot feature importance.


## Nonlinear Impact Model – Random Forest

In [19]:
if 'X_train' in globals() and y_train_reg is not None:
    rf = RandomForestRegressor(
        n_estimators=200,
        max_depth=10,
        random_state=42
    )

    rf.fit(X_train, y_train_reg)
    rf_pred = rf.predict(X_test)

    rmse_rf = np.sqrt(mean_squared_error(y_test_reg, rf_pred))
    r2_rf = r2_score(y_test_reg, rf_pred)

    print("Random Forest RMSE:", rmse_rf)
    print("Random Forest R^2:", r2_rf)
else:
    print("Regression target not available. Cannot train Random Forest.")

Regression target not available. Cannot train Random Forest.
